In [1]:
# Import standard libraries
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [2]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env import BalanceBotEnv
from rl.ppo_trainer import train, PPOConfig

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 1   # Single environment for now (so we can watch it train)

In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    total_timesteps = 200_000,     # Total simulation steps across all envs and iterations
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.002,              # Match MJCF opt.timestep for real-time rendering
)

In [5]:
# Build the environment
env = BalanceBotEnv(
    mjcf_path    = MJCF_PATH,
    render_mode  = "human",
)

# Wrap in RecordEpisodeStatistics so we can log episodic returns
env = gym.wrappers.RecordEpisodeStatistics(env)

# Wrap in SyncVectorEnv so it's compatible with train()
envs = gym.vector.SyncVectorEnv([lambda: env])

In [6]:
# Choo choo train!
agent = train(ppo_config, envs=envs)

SPS: 263
SPS: 277
SPS: 273
SPS: 276
SPS: 278
SPS: 278
SPS: 276
SPS: 276
SPS: 276
SPS: 276
SPS: 276
SPS: 273
SPS: 273
SPS: 272
SPS: 272
SPS: 273
SPS: 273
SPS: 274
SPS: 274
SPS: 274
SPS: 274
SPS: 275
SPS: 276
SPS: 276
SPS: 277
SPS: 277
SPS: 277
SPS: 278
SPS: 278
SPS: 278
SPS: 279
SPS: 278
SPS: 277
SPS: 277
SPS: 277
SPS: 278
SPS: 278
SPS: 278
SPS: 278
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777668207/checkpoint_iter0050.cleanrl_model
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 278
SPS: 278
SPS: 278
SPS: 278
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 279
SPS: 280
SPS: 279
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
SPS: 280
final model saved to 

NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.


Evaluation complete: mean return = nan over 10 episodes


In [7]:
# Close the environments
envs.close()

In [15]:
# %%%TEST: clean this up (i.e. load specific/best model, add "get device" function, etc.)
import torch
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent.eval()

# Reset the environment
obs, _ = envs.reset()
total_reward = 0
episode = 0
max_episodes = 3  # watch 3 episodes

while episode < max_episodes:
    step_start = time.time()

    # Get action from the trained agent (no gradient tracking needed)
    with torch.no_grad():
        action, _, _, _ = agent.get_action_and_value(
            torch.Tensor(obs).to(device)
        )

    # Step the environment
    obs, reward, terminated, truncated, infos = envs.step(action.cpu().numpy())
    total_reward += reward[0]

    # Render
    envs.envs[0].render()

    # Check if episode ended
    if terminated[0] or truncated[0]:
        print(f"Episode {episode + 1} ended — total reward: {total_reward:.2f}")
        total_reward = 0
        episode += 1
        obs, _ = envs.reset()

    # Real-time pacing
    while (time.time() - step_start) < 0.002:
        pass

Episode 1 ended — total reward: 9736.63
Episode 2 ended — total reward: 3936.71
Episode 3 ended — total reward: 903.54


In [13]:
# %%%TEST: Export training metrics as CSV. Make this into separate Python script?

import csv
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from pathlib import Path

# Point to a specific run directory
run_path = Path("/workspace/software/02-balance-bot-with-ppo/runs/BalanceBot-v0__balance-bot-ppo__42__1777668207")

# Load the event data
ea = EventAccumulator(run_path)
ea.Reload()

# See what metrics are available
all_tags = ea.Tags()['scalars']
print(f"Available metrics: {all_tags}")

# Split into training and eval metrics
train_tags = [t for t in all_tags if not t.startswith('eval/')]
eval_tags  = [t for t in all_tags if t.startswith('eval/')]

def export_csv(tags, output_path):
    """Export a list of scalar tags to a single CSV file."""
    if not tags:
        print(f"No tags to export for {output_path}")
        return

    # Collect all data: {tag: [(step, value), ...]}
    data = {}
    all_steps = set()
    for tag in tags:
        events = ea.Scalars(tag)
        data[tag] = {e.step: e.value for e in events}
        all_steps.update(data[tag].keys())

    # Write CSV with one column per metric, rows indexed by step
    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)

        # Header row
        writer.writerow(['step'] + tags)

        # One row per step, fill missing values with empty string
        for step in sorted(all_steps):
            row = [step] + [data[tag].get(step, '') for tag in tags]
            writer.writerow(row)

    print(f"Exported {len(tags)} metrics to {output_path}")

export_csv(train_tags, run_path / "training_metrics.csv")
export_csv(eval_tags,  run_path / "eval_metrics.csv")

Available metrics: ['charts/learning_rate', 'losses/value_loss', 'losses/policy_loss', 'losses/entropy', 'losses/old_approx_kl', 'losses/approx_kl', 'losses/clipfrac', 'losses/explained_variance', 'charts/SPS', 'eval/episodic_return']
Exported 9 metrics to /workspace/software/02-balance-bot-with-ppo/runs/BalanceBot-v0__balance-bot-ppo__42__1777668207/training_metrics.csv
Exported 1 metrics to /workspace/software/02-balance-bot-with-ppo/runs/BalanceBot-v0__balance-bot-ppo__42__1777668207/eval_metrics.csv
